# First 3 steps of non algebraicity proof

We starts with the rank-4 matroid on `{1,...,10}` specified by its 20 nonbases and uses SageMath's linear-subclass enumerator to perform these three application of Ingleton–Main lemma:

1. `{1,3}, {2,4}, {6,7}` and add element `11` — 1 valid extension.
2. `{4,6}, {5,8}, {9,10}` and add element `12` — 1 valid extension.
3. `{5,6}, {4,8}, {7,9}` and add element `13` — 3 valid extensions.

The first two application force a single matroid `M12`; the last application gives the three matroid `C_1, C_2, C_3`

In [1]:
from itertools import combinations
from sage.all import *
from sage.matroids.advanced import BasisMatroid

In [2]:
def im_extensions(M, generating_pairs, new_element):
    """
    Enumerate the proper Ingleton–Main extensions for three lines of M.
    """
    pairs = [frozenset(P) for P in generating_pairs]
    lines = [M.closure(P) for P in pairs]

    # Verify hypotheses of the Ingleton–Main lemma.
    assert all(M.rank(L) == 2 for L in lines), "Each set must span a line."
    assert all(M.rank(lines[i] | lines[j]) == 3
               for i, j in combinations(range(3), 2)), \
           "Every two lines must be coplanar."
    assert M.rank(frozenset().union(*lines)) == 4, \
           "The three lines must not all be coplanar."
    assert not set.intersection(*(set(L) for L in lines)), \
           "The three lines must not already have a common point."

    # Enumerates the linear subclasses compatible with the
    # condition that the new element be spanned by all three lines.
    subclasses = list(M.linear_subclasses(subsets=lines))
    candidates = [M._extension(new_element, C) for C in subclasses]
    valid = [N for N in candidates if len(N.loops()) == 0]#remove the loop extension

    return valid

## 1. The original matroid

In [3]:
NONBASES = [
    {1, 2, 3, 4},
    {1, 2, 5, 6},
    {1, 3, 6, 7},
    {2, 3, 5, 7},
    {2, 4, 6, 7},
    {2, 5, 8, 9},
    {2, 5, 8, 10},
    {2, 5, 9, 10},
    {2, 8, 9, 10},
    {3, 4, 5, 6},
    {3, 4, 5, 8},
    {3, 4, 6, 8},
    {3, 5, 6, 8},
    {4, 5, 6, 8},
    {4, 5, 7, 10},
    {4, 6, 9, 10},
    {4, 7, 8, 9},
    {5, 6, 7, 9},
    {5, 8, 9, 10},
    {6, 7, 8, 10},
]

M10 = BasisMatroid(
    groundset=frozenset(range(1, 11)),
    nonbases=[frozenset(X) for X in NONBASES],
)

assert M10.is_valid()
assert M10.is_simple()
assert M10.full_rank() == 4
assert M10.bases_count() == 190       # binomial(10,4) - 20

print(M10)
print("rank =", M10.full_rank())
print("bases =", M10.bases_count())
print("nonbases =", len(M10.nonbases()))

Matroid of rank 4 on 10 elements with 190 bases
rank = 4
bases = 190
nonbases = 20


## 2. First Ingleton–Main application: add 11 on `{1,3}`, `{2,4}`, `{6,7}`

There is exactly one proper extension `M11`.

In [4]:
step1 = im_extensions(
    M10,
    [{1, 3}, {2, 4}, {6, 7}],
    new_element=11,
)

assert len(step1) == 1
M11 = step1[0]
print("M11 bases:", M11.bases_count())
print("M11 nonbases:", len(M11.nonbases()))

M11 bases: 282
M11 nonbases: 48


## 3. Second Ingleton–Main application: add 12 on `{4,6}`, `{5,8}`, `{9,10}`

Again there is exactly one proper extension `M12`.

In [5]:
step2 = im_extensions(
    M11,
    [{4, 6}, {5, 8}, {9, 10}],
    new_element=12,
)

assert len(step2) == 1
M12 = step2[0]
print("M12 bases:", M12.bases_count())
print("M12 nonbases:", len(M12.nonbases()))

M12 bases: 404
M12 nonbases: 91


## 4. Third Ingleton–Main application: add 13 on `{5,6}`, `{4,8}`, and `{7,9}`

 There are 3 valid extensions.

In [6]:
step3 = im_extensions(
    M12,
    [{5, 6}, {4, 8}, {7, 9}],
    new_element=13,
)

M13_children = sorted(
    step3,
    key=lambda N: -N.bases_count(),
)

assert len(M13_children) == 3

for i, N in enumerate(M13_children, start=1):
    print(
        f"C_{i}: "
        f"{N.bases_count()} bases, {len(N.nonbases())} nonbases"
    )

C_1: 571 bases, 144 nonbases
C_2: 570 bases, 145 nonbases
C_3: 568 bases, 147 nonbases


In [7]:
def revlex_basis_encoding(M, groundset_order=None):
    """
    Return M's Oscar/Polymake REVLEX_BASIS_ENCODING.

    '*' means basis and '0' means nonbasis.

    groundset_order determines the correspondence with Oscar's labels
    1,...,n. For example, use range(1, n + 1) for integer-labelled
    matroids.
    """
    if groundset_order is None:
        try:
            E = sorted(M.groundset())
        except TypeError:
            raise ValueError(
                "The ground-set labels are not sortable; "
                "supply groundset_order explicitly."
            )
    else:
        E = list(groundset_order)

    if len(E) != len(M.groundset()) or frozenset(E) != M.groundset():
        raise ValueError(
            "groundset_order must contain every ground-set element exactly once."
        )

    n = len(E)
    r = int(M.full_rank())

    # Generate r-subsets in ascending colexicographic order,
    # which is the REVLEX order used by Oscar/Polymake.
    def colex_indices(n, r):
        if r == 0:
            yield ()
            return

        # Group subsets by their largest element.
        for last in range(r - 1, n):
            for prefix in colex_indices(last, r - 1):
                yield prefix + (last,)

    is_basis = M.is_basis

    return "".join(
        "*" if is_basis(frozenset(E[i] for i in indices)) else "0"
        for indices in colex_indices(n, r)
    )
for i, N in enumerate(M13_children, start=1):
    print(
        f"C_{i}: "
        f"{revlex_basis_encoding(N)}"
    )

C_1: 0********0****0******0****0**0**************0*****0**00*************************************************0*******0**********0*******************************0************0************0*******0*****0********0**0**0000*0**0**0**0******0**0*****00000*0**0***************0*0**0*********0****00********0**0***************0*****00*****************0***000**00****0******0*0******00000**000****0********0*****0********0**0*********0*****0********0**0**00000000*0**0******0*0**0*0*0****0******************0****0****00****00000**************0***000**00**000***0****************0000000***0**0**************0*********0*********0**0**0*********0****00*0*0*******000***************0**00**000********0000*******0*******0*0**********
C_2: 0********0****0******0****0**0**************0*****0**00*************************************************0*******0**********0*******************************0************0************0*******0*****0********0**0**0000*0**0**0**0******0**0*****00000*0**0***************0*0**0***

In [8]:
print("Compatible linear-subclass counts: 2 -> 2 -> 4")
print("After removing the loop:          1 -> 1 -> 3")
print("Forced chain: M10 --(add 11)--> M11 --(add 12)--> M12")
print("Final branch: M12 --(add 13)--> three distinct M13 children")
print("Child basis counts:", [N.bases_count() for N in M13_children])

Compatible linear-subclass counts: 2 -> 2 -> 4
After removing the loop:          1 -> 1 -> 3
Forced chain: M10 --(add 11)--> M11 --(add 12)--> M12
Final branch: M12 --(add 13)--> three distinct M13 children
Child basis counts: [571, 570, 568]
